In [ ]:
# Cell 0: Install dependencies
!pip install transformers torch sentencepiece -q


In [ ]:
# Cell 1: Imports
import os
import gc
import time
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.nn.utils.rnn import pad_sequence
from transformers import XLMRobertaModel, XLMRobertaTokenizer, AutoTokenizer
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")


✓ Libraries imported


In [ ]:
# Cell 2: HuggingFace Login via Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✓ HuggingFace logged in via Kaggle Secret")


✓ HuggingFace logged in via Kaggle Secret


In [ ]:
# Cell 3: Configuration
# Load XLM-R tokenizer — same as training
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
print(f"✓ Tokenizer loaded  |  vocab size: {tokenizer.vocab_size:,}")

# All 15 language pairs (used for both training AND evaluation — no zero-shot split for demo model)
ALL_LANGUAGE_PAIRS = [
    ("Cantonese", "English"),
    ("Arabic", "English"),
    ("Philippines", "English"),
    ("German", "French"),
    ("Chinese", "English"),
    ("Vietnamese", "English"),
    ("Malay", "English"),
    ("Japanese", "English"),
    ("Hindi", "English"),
    ("Korean", "English"),
    ("Spanish", "English"),
    ("French", "English"),
    ("Russian", "English"),
    ("Italian", "English"),
    ("German", "English"),
]

print(f"Training on ALL {len(ALL_LANGUAGE_PAIRS)} language pairs (no zero-shot holdout)")


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded  |  vocab size: 250,002
Training on ALL 15 language pairs (no zero-shot holdout)


In [ ]:
# Cell 4: Load pre-processed pickle from Kaggle Dataset
# The dataset 'cs6140-codeswitch-data' contains all_15pairs_stats_xlmr.pkl
# (LID + subword alignment + labels already done with XLM-R tokenizer)

PICKLE_CANDIDATES = [
    '/kaggle/input/cs6140-codeswitch-data/all_15pairs_stats_xlmr.pkl',
    '/kaggle/input/cs6140-codeswitch-data/all_15pairs_stats.pkl',
    '/kaggle/input/datasets/yijiyimengchen/cs6140-codeswitch-data/all_15pairs_stats_xlmr.pkl',
]

PICKLE_PATH = None
for path in PICKLE_CANDIDATES:
    if os.path.exists(path):
        PICKLE_PATH = path
        break

if PICKLE_PATH is None:
    print("ERROR: No pickle found. Searching /kaggle/input/ ...")
    for root, dirs, files in os.walk('/kaggle/input/'):
        for f in files:
            if f.endswith('.pkl'):
                print(f"  Found: {os.path.join(root, f)}")
    raise FileNotFoundError("Adjust PICKLE_CANDIDATES above to match the actual path.")

print(f"Loading pickle from: {PICKLE_PATH}")
with open(PICKLE_PATH, 'rb') as f:
    all_stats = pickle.load(f)

print(f"✓ Loaded {len(all_stats)} language pairs:")
for k in all_stats:
    n_samples = len(all_stats[k]['processed_samples'])
    print(f"  {k:<25} {n_samples:>6} samples")


Loading pickle from: /kaggle/input/datasets/yijiyimengchen/cs6140-codeswitch-data/all_15pairs_stats_xlmr.pkl
✓ Loaded 15 language pairs:
  Cantonese-English           6000 samples
  Arabic-English              6000 samples
  Philippines-English         6000 samples
  German-French               6000 samples
  Chinese-English             6000 samples
  Vietnamese-English          6000 samples
  Malay-English               6000 samples
  Japanese-English            6000 samples
  Hindi-English               6000 samples
  Korean-English              6000 samples
  Spanish-English             6000 samples
  French-English              6000 samples
  Russian-English             6000 samples
  Italian-English             6000 samples
  German-English              6000 samples


In [ ]:
# Cell 5: Training hyperparameters

MAX_LEN      = 256
BATCH_SIZE   = 32
NUM_EPOCHS   = 4
BASE_LR      = 1e-5
HEAD_LR      = BASE_LR * 50   # 5e-4
WARMUP_RATIO = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f"✓ Device: {device}")
print(f"  GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"    GPU {i}: {torch.cuda.get_device_name(i)}  |  free: {free/1e9:.1f} GB")

print(f"\nHyperparameters:")
print(f"  MAX_LEN      = {MAX_LEN}")
print(f"  BATCH_SIZE   = {BATCH_SIZE}")
print(f"  NUM_EPOCHS   = {NUM_EPOCHS}")
print(f"  BASE_LR      = {BASE_LR}")
print(f"  HEAD_LR      = {HEAD_LR}")
print(f"  WARMUP_RATIO = {WARMUP_RATIO}")


✓ Device: cuda
  GPUs available: 2
    GPU 0: Tesla T4  |  free: 15.5 GB
    GPU 1: Tesla T4  |  free: 15.5 GB

Hyperparameters:
  MAX_LEN      = 256
  BATCH_SIZE   = 32
  NUM_EPOCHS   = 4
  BASE_LR      = 1e-05
  HEAD_LR      = 0.0005
  WARMUP_RATIO = 0.1


In [ ]:
# Cell 6: Dataset + Model (Full FT + Causal Mask)

xlm_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")


class XLMRCodeSwitchDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=MAX_LEN):
        self.items = []
        for s in samples:
            ids = tokenizer.convert_tokens_to_ids(s['tokens'])
            ids = [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id]

            sw_raw  = [x if x != -1 else -100 for x in s['y_switch']]
            dur_raw = [x if x != -1 else -100 for x in s['y_duration']]
            sw  = [-100] + sw_raw  + [-100]
            dur = [-100] + dur_raw + [-100]

            ids = ids[:max_len]
            n = len(ids)
            if n < 2:
                continue

            label_len = n - 1
            sw  = sw[:label_len]
            dur = dur[:label_len]

            if len(sw) != label_len or len(dur) != label_len:
                continue

            self.items.append({
                'input_ids':  torch.tensor(ids, dtype=torch.long),
                'y_switch':   torch.tensor(sw,  dtype=torch.long),
                'y_duration': torch.tensor(dur, dtype=torch.long),
            })

    def __len__(self):        return len(self.items)
    def __getitem__(self, i): return self.items[i]


def xlmr_collate(batch):
    pad_id = xlm_tokenizer.pad_token_id
    ids  = pad_sequence([b['input_ids']  for b in batch], batch_first=True, padding_value=pad_id)
    sw   = pad_sequence([b['y_switch']   for b in batch], batch_first=True, padding_value=-100)
    dur  = pad_sequence([b['y_duration'] for b in batch], batch_first=True, padding_value=-100)
    mask = (ids != pad_id).long()
    return {'input_ids': ids, 'attention_mask': mask, 'y_switch': sw, 'y_duration': dur}


def build_datasets(all_stats, pairs, tokenizer, train_ratio=0.8):
    """Build train/val from ALL pairs."""
    train_sets, val_sets = [], []
    print("Building datasets...")
    for lang1, lang2 in pairs:
        key = f"{lang1}-{lang2}"
        if key not in all_stats or not all_stats[key]['processed_samples']:
            print(f"  ⚠️  {key} not found, skipping.")
            continue
        samples  = all_stats[key]['processed_samples']
        split_at = int(len(samples) * train_ratio)
        tr = XLMRCodeSwitchDataset(samples[:split_at], tokenizer)
        va = XLMRCodeSwitchDataset(samples[split_at:],  tokenizer)
        print(f"  {key:<25} total={len(samples):>6}  train={len(tr):>6}  val={len(va):>6}")
        train_sets.append(tr); val_sets.append(va)
    return ConcatDataset(train_sets), ConcatDataset(val_sets)


# ─── Causal Mask (identical to 8-pair version) ───────────────────────────
def make_causal_mask(attention_mask):
    """
    Construct a 4D additive causal attention mask for XLM-R encoder.

    Input:
        attention_mask: [B, L] — 1 for real tokens, 0 for padding
    Output:
        extended_mask:  [B, 1, L, L] — additive, with -inf blocking future
                        tokens and padding; 0 allowing attendance.
    """
    B, L = attention_mask.shape
    device = attention_mask.device
    dtype  = torch.float32

    # 1D padding mask → 2D
    pad_mask = attention_mask[:, None, None, :].to(dtype)  # [B, 1, 1, L]
    pad_mask = (1.0 - pad_mask) * torch.finfo(dtype).min   # 0 → 0, 1 → -inf

    # Causal mask: upper-triangular above diagonal = -inf
    causal = torch.triu(torch.ones((L, L), dtype=dtype, device=device), diagonal=1)
    causal = causal * torch.finfo(dtype).min
    causal = causal[None, None, :, :]   # [1, 1, L, L]

    return pad_mask + causal   # broadcast → [B, 1, L, L]


# ─── Model (Full FT + Causal Mask) ────────────────────────────────────────

class CausalXLMRCodeSwitchPredictor(nn.Module):
    def __init__(self, model_name="xlm-roberta-base",
                 dropout=0.1, freeze_encoder=False):
        super().__init__()
        print(f"Loading {model_name}...")
        self.encoder = XLMRobertaModel.from_pretrained(model_name)
        d = self.encoder.config.hidden_size

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
            print("  Encoder FROZEN — only heads are trainable")
        else:
            print("  Encoder UNFROZEN — full fine-tuning")

        def _head(out):
            return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(d, d // 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d // 2, out),
            )

        self.switch_head   = _head(2)
        self.duration_head = _head(3)

        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"✓ Model ready  |  total={total:,}  trainable={train:,}")

    def forward(self, input_ids, attention_mask=None):
        # Build 4D causal mask (this is what makes prediction truly anticipatory)
        if attention_mask is not None:
            extended_mask = make_causal_mask(attention_mask)
        else:
            extended_mask = None

        out = self.encoder(
            input_ids=input_ids,
            attention_mask=extended_mask,
            return_dict=True,
        )
        # Predict at position t → label for t+1, so slice off last position
        hidden = out.last_hidden_state[:, :-1, :]
        return self.switch_head(hidden), self.duration_head(hidden)


# ─── Class weights ────────────────────────────────────────────────────────
def compute_class_weights(loader, num_sw=2, num_dur=3):
    sw_counts  = torch.zeros(num_sw)
    dur_counts = torch.zeros(num_dur)
    for batch in loader:
        y_sw  = batch['y_switch'].reshape(-1)
        y_dur = batch['y_duration'].reshape(-1)
        for c in range(num_sw):
            sw_counts[c] += (y_sw == c).sum()
        for c in range(num_dur):
            dur_counts[c] += (y_dur == c).sum()
    sw_w  = (1.0 / sw_counts);  sw_w  /= sw_w.min()
    dur_w = (1.0 / dur_counts); dur_w /= dur_w.min()
    print(f"  Switch  counts : {sw_counts.long().tolist()}")
    print(f"  Switch  weights: {[f'{w:.4f}' for w in sw_w.tolist()]}")
    print(f"  Duration counts: {dur_counts.long().tolist()}")
    print(f"  Duration weights: {[f'{w:.4f}' for w in dur_w.tolist()]}")
    return sw_w, dur_w


# ─── Warmup + Cosine scheduler ────────────────────────────────────────────
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.01):
        self.opt           = optimizer
        self.warmup_steps  = warmup_steps
        self.total_steps   = total_steps
        self.min_lr_ratio  = min_lr_ratio
        self.base_lrs      = [g['lr'] for g in optimizer.param_groups]
        self.step_count    = 0

    def step(self):
        self.step_count += 1
        for i, g in enumerate(self.opt.param_groups):
            base = self.base_lrs[i]
            if self.step_count < self.warmup_steps:
                lr = base * (self.step_count / max(1, self.warmup_steps))
            else:
                progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
                cos_factor = 0.5 * (1 + np.cos(np.pi * progress))
                lr = base * (self.min_lr_ratio + (1 - self.min_lr_ratio) * cos_factor)
            g['lr'] = lr

    def get_last_lr(self):
        return [g['lr'] for g in self.opt.param_groups]

    def state_dict(self):
        return {'step_count': self.step_count, 'base_lrs': self.base_lrs}

    def load_state_dict(self, sd):
        self.step_count = sd['step_count']
        self.base_lrs   = sd['base_lrs']


# ─── Train / Eval loops ───────────────────────────────────────────────────

def train_epoch(model, loader, optimiser, scheduler, device, sw_criterion, dur_criterion):
    model.train()
    sw_losses, dur_losses = [], []
    for batch in tqdm(loader, desc="  train", leave=False):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        y_sw  = batch['y_switch'].to(device)
        y_dur = batch['y_duration'].to(device)

        sw_logits, dur_logits = model(ids, mask)

        loss_sw = sw_criterion(sw_logits.reshape(-1, 2), y_sw.reshape(-1))

        dur_flat  = y_dur.reshape(-1)
        dlog_flat = dur_logits.reshape(-1, 3)
        valid     = dur_flat >= 0
        loss_dur  = (
            dur_criterion(dlog_flat[valid], dur_flat[valid])
            if valid.any() else torch.tensor(0.0, device=device)
        )

        loss = loss_sw + 1.0 * loss_dur

        optimiser.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()

        sw_losses.append(loss_sw.item())
        dur_losses.append(loss_dur.item())

    return {'loss_sw': np.mean(sw_losses), 'loss_dur': np.mean(dur_losses)}


def evaluate(model, loader, device, sw_criterion=None, dur_criterion=None):
    model.eval()
    sw_pred, sw_true   = [], []
    dur_pred, dur_true = [], []
    val_sw_losses, val_dur_losses = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            y_sw  = batch['y_switch'].to(device)
            y_dur = batch['y_duration'].to(device)
            sw_logits, dur_logits = model(ids, mask)

            if sw_criterion is not None:
                val_sw_losses.append(sw_criterion(sw_logits.reshape(-1, 2), y_sw.reshape(-1)).item())
            if dur_criterion is not None:
                dur_flat  = y_dur.reshape(-1)
                dlog_flat = dur_logits.reshape(-1, 3)
                valid     = dur_flat >= 0
                if valid.any():
                    val_dur_losses.append(dur_criterion(dlog_flat[valid], dur_flat[valid]).item())

            sw_hat  = sw_logits.argmax(-1).cpu()
            dur_hat = dur_logits.argmax(-1).cpu()
            y_sw_cpu  = y_sw.cpu()
            y_dur_cpu = y_dur.cpu()
            B, L1 = sw_hat.shape
            for b in range(B):
                for t in range(L1):
                    if y_sw_cpu[b, t].item() != -100:
                        sw_pred.append(sw_hat[b, t].item())
                        sw_true.append(y_sw_cpu[b, t].item())
                    if y_dur_cpu[b, t].item() not in (-1, -100):
                        dur_pred.append(dur_hat[b, t].item())
                        dur_true.append(y_dur_cpu[b, t].item())
    sw_f1  = f1_score(sw_true,  sw_pred,  average='macro', zero_division=0)
    dur_f1 = f1_score(dur_true, dur_pred, average='macro', zero_division=0) if dur_true else 0.0
    out = {
        'switch_f1':       sw_f1,
        'duration_f1':     dur_f1,
        'switch_report':   classification_report(sw_true, sw_pred,
                               target_names=['no-switch', 'switch'], zero_division=0),
        'duration_report': classification_report(dur_true, dur_pred,
                               target_names=['small', 'medium', 'large'], zero_division=0)
                           if dur_true else "No duration preds.",
    }
    if val_sw_losses:  out['val_loss_sw']  = np.mean(val_sw_losses)
    if val_dur_losses: out['val_loss_dur'] = np.mean(val_dur_losses)
    return out


def evaluate_per_pair(model, all_stats, pairs, tokenizer, device):
    """Per-pair F1 — for sanity check during training."""
    results = {}
    for lang1, lang2 in pairs:
        key = f"{lang1}-{lang2}"
        if key not in all_stats or not all_stats[key]['processed_samples']:
            continue
        samples  = all_stats[key]['processed_samples']
        split_at = int(len(samples) * 0.8)
        val_ds   = XLMRCodeSwitchDataset(samples[split_at:], tokenizer)
        if len(val_ds) == 0:
            continue
        ldr = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=xlmr_collate)
        results[key] = evaluate(model, ldr, device)
    return results

print("✓ Model + training utilities defined")


✓ Model + training utilities defined


In [ ]:
# Cell 7: GPU memory check
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  {torch.cuda.get_device_name(i)}  |  free: {free/1e9:.1f} GB / total: {total/1e9:.1f} GB")
    if torch.cuda.device_count() >= 2:
        print("✓ DataParallel will be enabled (effective batch = BATCH_SIZE × num_gpus)")
    else:
        print("  Single GPU training")
else:
    print("  No GPU available — will train on CPU (very slow)")


Number of GPUs: 2
  Tesla T4  |  free: 15.5 GB / total: 15.6 GB
  Tesla T4  |  free: 15.5 GB / total: 15.6 GB
✓ DataParallel will be enabled (effective batch = BATCH_SIZE × num_gpus)


In [ ]:
# Cell 8: Main training pipeline — Causal Full FT on ALL 15 pairs
# Kaggle additions:
#   - DataParallel across 2 T4s
#   - Per-epoch checkpoint to /kaggle/working/ckpt_latest_demo_causal.pt
#   - Resume from checkpoint if exists (set RESUME=True)
#   - Time budget guard

CKPT_LATEST  = "/kaggle/working/ckpt_latest_demo_causal.pt"
BEST_MODEL   = "/kaggle/working/best_xlmr_demo_15pairs_causal.pt"
MAX_TRAIN_HOURS = 3.0   # plenty for 4 epochs on 15 pairs
RESUME = True


def run_pipeline(all_stats, tokenizer, epochs=NUM_EPOCHS):
    gc.collect(); torch.cuda.empty_cache()

    # Train/val from ALL 15 pairs
    ALL_PAIRS_TUPLES = [(k.split('-')[0], k.split('-')[1]) for k in all_stats.keys()]
    train_ds, val_ds = build_datasets(all_stats, ALL_PAIRS_TUPLES, tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  collate_fn=xlmr_collate, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=xlmr_collate, num_workers=2)

    print("\nComputing class weights from training data...")
    sw_w, dur_w = compute_class_weights(train_loader)
    sw_w  = sw_w.to(device)
    dur_w = dur_w.to(device)

    sw_criterion  = nn.CrossEntropyLoss(weight=sw_w,  ignore_index=-100)
    dur_criterion = nn.CrossEntropyLoss(weight=dur_w)

    # Build model + wrap DataParallel
    model = CausalXLMRCodeSwitchPredictor(freeze_encoder=False).to(device)
    use_dp = torch.cuda.device_count() >= 2
    if use_dp:
        model = nn.DataParallel(model)
        print(f"✓ DataParallel enabled across {torch.cuda.device_count()} GPUs")
    else:
        print("  Single GPU (no DataParallel)")

    def unwrap(m):
        return m.module if isinstance(m, nn.DataParallel) else m

    optimiser = torch.optim.AdamW([
        {'params': unwrap(model).encoder.parameters(),       'lr': BASE_LR},
        {'params': unwrap(model).switch_head.parameters(),   'lr': HEAD_LR},
        {'params': unwrap(model).duration_head.parameters(), 'lr': HEAD_LR},
    ], weight_decay=0.01)

    total_steps  = len(train_loader) * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = WarmupCosineScheduler(optimiser, warmup_steps, total_steps)

    # Resume if checkpoint exists
    start_epoch = 1
    history     = []
    best_sw_f1  = 0.0

    if RESUME and os.path.exists(CKPT_LATEST):
        print(f"\n→ Found checkpoint: {CKPT_LATEST}")
        ckpt = torch.load(CKPT_LATEST, map_location=device, weights_only=False)
        unwrap(model).load_state_dict(ckpt['model_state'])
        optimiser.load_state_dict(ckpt['optim_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        best_sw_f1  = ckpt['best_sw_f1']
        history     = ckpt['history']
        start_epoch = ckpt['epoch'] + 1
        print(f"✓ Resumed from epoch {ckpt['epoch']}, continuing from epoch {start_epoch}")
    else:
        print("\n→ Fresh training")

    print(f"\n  Mode: CAUSAL FULL FINE-TUNING (15 pairs, demo backbone)")
    print(f"  Attention: position t sees only 0..t (no future)")
    print(f"  Encoder LR: {BASE_LR}  |  Head LR: {HEAD_LR}")
    print(f"  Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")
    print(f"  Train pairs: {len(ALL_PAIRS_TUPLES)}  (all-pairs training, 80/20 split within each)")
    print(f"  Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")
    print(f"  Training on: {device}  |  Time budget: {MAX_TRAIN_HOURS}h\n")

    start_time = time.time()

    for epoch in range(start_epoch, epochs + 1):
        losses  = train_epoch(model, train_loader, optimiser, scheduler,
                              device, sw_criterion, dur_criterion)
        metrics = evaluate(model, val_loader, device,
                           sw_criterion, dur_criterion)
        pair_f1s = {k: v['switch_f1'] for k, v in
                    evaluate_per_pair(model, all_stats, ALL_PAIRS_TUPLES,
                                      tokenizer, device).items()}

        current_lrs = scheduler.get_last_lr()
        elapsed_h = (time.time() - start_time) / 3600

        history.append({
            'epoch':        epoch,
            'loss_sw':      losses['loss_sw'],
            'loss_dur':     losses['loss_dur'],
            'val_loss_sw':  metrics.get('val_loss_sw', 0.0),
            'val_loss_dur': metrics.get('val_loss_dur', 0.0),
            'switch_f1':    metrics['switch_f1'],
            'duration_f1':  metrics['duration_f1'],
            'pair_f1s':     pair_f1s,
            'encoder_lr':   current_lrs[0],
            'elapsed_h':    elapsed_h,
        })

        print(f"Epoch {epoch:02d} | "
              f"train_sw={losses['loss_sw']:.4f}  train_dur={losses['loss_dur']:.4f} | "
              f"val_sw={metrics.get('val_loss_sw', 0):.4f}  "
              f"val_dur={metrics.get('val_loss_dur', 0):.4f} | "
              f"switch_F1={metrics['switch_f1']:.4f}  "
              f"duration_F1={metrics['duration_f1']:.4f}  | "
              f"enc_lr={current_lrs[0]:.2e}  |  elapsed={elapsed_h:.2f}h")
        for k, v in pair_f1s.items():
            print(f"         {k:<25} switch_F1={v:.4f}")

        # Per-epoch checkpoint
        torch.save({
            'epoch': epoch,
            'model_state':     unwrap(model).state_dict(),
            'optim_state':     optimiser.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_sw_f1':      best_sw_f1,
            'history':         history,
        }, CKPT_LATEST)

        # Save best model
        if metrics['switch_f1'] > best_sw_f1:
            best_sw_f1 = metrics['switch_f1']
            torch.save(unwrap(model).state_dict(), BEST_MODEL)
            print(f"  ✓ New best switch F1: {best_sw_f1:.4f} — saved to {BEST_MODEL}")

        # Time budget guard
        if elapsed_h > MAX_TRAIN_HOURS:
            print(f"\n⚠ Reached {MAX_TRAIN_HOURS}h budget at epoch {epoch}. Stopping early.")
            break

    # Final eval with best checkpoint
    unwrap(model).load_state_dict(torch.load(BEST_MODEL, map_location=device, weights_only=False))
    final = evaluate(model, val_loader, device, sw_criterion, dur_criterion)
    print("\n===== Final Evaluation (Causal Full FT Demo Model, best checkpoint) =====")
    print("-- Switch --\n",   final['switch_report'])
    print("-- Duration --\n", final['duration_report'])

    return model, final, history


xlmr_demo_model, xlmr_demo_final, xlmr_demo_history = run_pipeline(
    all_stats, xlm_tokenizer, epochs=NUM_EPOCHS
)


Building datasets...
  Cantonese-English         total=  6000  train=  4800  val=  1200
  Arabic-English            total=  6000  train=  4800  val=  1200
  Philippines-English       total=  6000  train=  4800  val=  1200
  German-French             total=  6000  train=  4800  val=  1200
  Chinese-English           total=  6000  train=  4800  val=  1200
  Vietnamese-English        total=  6000  train=  4800  val=  1200
  Malay-English             total=  6000  train=  4800  val=  1200
  Japanese-English          total=  6000  train=  4800  val=  1200
  Hindi-English             total=  6000  train=  4800  val=  1200
  Korean-English            total=  6000  train=  4800  val=  1200
  Spanish-English           total=  6000  train=  4800  val=  1200
  French-English            total=  6000  train=  4800  val=  1200
  Russian-English           total=  6000  train=  4800  val=  1200
  Italian-English           total=  6000  train=  4800  val=  1200
  German-English            total=  6000 

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoder UNFROZEN — full fine-tuning
✓ Model ready  |  total=278,636,165  trainable=278,636,165
✓ DataParallel enabled across 2 GPUs

→ Fresh training

  Mode: CAUSAL FULL FINE-TUNING (15 pairs, demo backbone)
  Attention: position t sees only 0..t (no future)
  Encoder LR: 1e-05  |  Head LR: 0.0005
  Total steps: 9000  |  Warmup steps: 900
  Train pairs: 15  (all-pairs training, 80/20 split within each)
  Train batches: 2250  |  Val batches: 563
  Training on: cuda  |  Time budget: 3.0h



Epoch 01 | train_sw=0.4210  train_dur=0.9445 | val_sw=0.3380  val_dur=0.8414 | switch_F1=0.7381  duration_F1=0.5607  | enc_lr=9.34e-06  |  elapsed=0.66h
         Cantonese-English         switch_F1=0.7235
         Arabic-English            switch_F1=0.7302
         Philippines-English       switch_F1=0.6876
         German-French             switch_F1=0.7336
         Chinese-English           switch_F1=0.7451
         Vietnamese-English        switch_F1=0.7261
         Malay-English             switch_F1=0.6830
         Japanese-English          switch_F1=0.7561
         Hindi-English             switch_F1=0.7600
         Korean-English            switch_F1=0.7910
         Spanish-English           switch_F1=0.6187
         French-English            switch_F1=0.6969
         Russian-English           switch_F1=0.7360
         Italian-English           switch_F1=0.7507
         German-English            switch_F1=0.7303
  ✓ New best switch F1: 0.7381 — saved to /kaggle/working/best_xlmr

Epoch 02 | train_sw=0.3404  train_dur=0.8626 | val_sw=0.3156  val_dur=0.8183 | switch_F1=0.7636  duration_F1=0.5652  | enc_lr=5.91e-06  |  elapsed=1.32h
         Cantonese-English         switch_F1=0.7556
         Arabic-English            switch_F1=0.7524
         Philippines-English       switch_F1=0.7109
         German-French             switch_F1=0.7598
         Chinese-English           switch_F1=0.7806
         Vietnamese-English        switch_F1=0.7528
         Malay-English             switch_F1=0.7048
         Japanese-English          switch_F1=0.7894
         Hindi-English             switch_F1=0.7909
         Korean-English            switch_F1=0.8124
         Spanish-English           switch_F1=0.6619
         French-English            switch_F1=0.7233
         Russian-English           switch_F1=0.7613
         Italian-English           switch_F1=0.7722
         German-English            switch_F1=0.7579
  ✓ New best switch F1: 0.7636 — saved to /kaggle/working/best_xlmr

Epoch 03 | train_sw=0.3246  train_dur=0.8387 | val_sw=0.3086  val_dur=0.8063 | switch_F1=0.7637  duration_F1=0.5763  | enc_lr=1.87e-06  |  elapsed=1.98h
         Cantonese-English         switch_F1=0.7542
         Arabic-English            switch_F1=0.7526
         Philippines-English       switch_F1=0.7128
         German-French             switch_F1=0.7659
         Chinese-English           switch_F1=0.7793
         Vietnamese-English        switch_F1=0.7573
         Malay-English             switch_F1=0.7078
         Japanese-English          switch_F1=0.7886
         Hindi-English             switch_F1=0.7934
         Korean-English            switch_F1=0.8113
         Spanish-English           switch_F1=0.6507
         French-English            switch_F1=0.7216
         Russian-English           switch_F1=0.7624
         Italian-English           switch_F1=0.7751
         German-English            switch_F1=0.7583
  ✓ New best switch F1: 0.7637 — saved to /kaggle/working/best_xlmr

Epoch 04 | train_sw=0.3184  train_dur=0.8263 | val_sw=0.3068  val_dur=0.8047 | switch_F1=0.7658  duration_F1=0.5771  | enc_lr=1.00e-07  |  elapsed=2.65h
         Cantonese-English         switch_F1=0.7579
         Arabic-English            switch_F1=0.7558
         Philippines-English       switch_F1=0.7122
         German-French             switch_F1=0.7654
         Chinese-English           switch_F1=0.7841
         Vietnamese-English        switch_F1=0.7577
         Malay-English             switch_F1=0.7079
         Japanese-English          switch_F1=0.7923
         Hindi-English             switch_F1=0.7925
         Korean-English            switch_F1=0.8127
         Spanish-English           switch_F1=0.6587
         French-English            switch_F1=0.7273
         Russian-English           switch_F1=0.7647
         Italian-English           switch_F1=0.7777
         German-English            switch_F1=0.7587
  ✓ New best switch F1: 0.7658 — saved to /kaggle/working/best_xlmr

In [ ]:
# Cell 9: List output files (persist as Notebook Output after Commit)
import os, json

print("=== Files in /kaggle/working/ ===")
for fname in sorted(os.listdir('/kaggle/working/')):
    fpath = os.path.join('/kaggle/working/', fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"  {fname:<50} {size_mb:>8.2f} MB")

# Save history as JSON
history_path = '/kaggle/working/xlmr_demo_history.json'
with open(history_path, 'w') as f:
    clean = []
    for h in xlmr_demo_history:
        clean_h = {k: (v if not isinstance(v, dict) else
                       {kk: float(vv) for kk, vv in v.items()})
                   for k, v in h.items()}
        clean.append(clean_h)
    json.dump(clean, f, indent=2, default=float)
print(f"\n✓ History saved: {history_path}")

# Save summary
summary = {
    'model': 'xlm-roberta-base',
    'mode':  'causal_full_fine_tuning_demo_backbone',
    'trained_on': 'all_15_pairs (80/20 within-pair split)',
    'hyperparams': {
        'MAX_LEN':    MAX_LEN,
        'BATCH_SIZE': BATCH_SIZE,
        'NUM_EPOCHS': NUM_EPOCHS,
        'BASE_LR':    BASE_LR,
        'HEAD_LR':    HEAD_LR,
        'WARMUP_RATIO': WARMUP_RATIO,
    },
    'final_switch_f1':   float(xlmr_demo_final['switch_f1']),
    'final_duration_f1': float(xlmr_demo_final['duration_f1']),
    'n_epochs_run':      len(xlmr_demo_history),
}
with open('/kaggle/working/xlmr_demo_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Summary saved: /kaggle/working/xlmr_demo_summary.json")

print("\n=== DONE ===")
print("Next step: download best_xlmr_demo_15pairs_causal.pt to your local machine,")
print("then upload to Google Drive at /MyDrive/CS6140/best_xlmr_demo_15pairs_causal.pt")


=== Files in /kaggle/working/ ===
  __notebook__.ipynb                                     1.59 MB
  best_xlmr_demo_15pairs_causal.pt                    1063.00 MB
  ckpt_latest_demo_causal.pt                          3189.00 MB

✓ History saved: /kaggle/working/xlmr_demo_history.json
✓ Summary saved: /kaggle/working/xlmr_demo_summary.json

=== DONE ===
Next step: download best_xlmr_demo_15pairs_causal.pt to your local machine,
then upload to Google Drive at /MyDrive/CS6140/best_xlmr_demo_15pairs_causal.pt
